In [ ]:
import os
import sys

sys.path.insert(0, os.path.dirname(os.getcwd()))

In [ ]:
import numpy as np
import pandas as pd
from datasets import load_dataset
from transformers import AutoProcessor

from src.constants import WESTBROOK_DATASET_ACCENT_MAP
from src.model import register_whisper_accent
from src.train.train import ModelArguments, processor_init
from src.train.dataset import DataCollatorSpeechSeq2SeqWithPadding, WhisperDataset

register_whisper_accent()

In [ ]:
def get_accent_counts():
    dataset_path = "westbrook/English_Accent_DataSet"
    dataset = load_dataset(dataset_path)
    li = []
    for split in dataset:
        ids, counts = np.unique(np.array(dataset[split]["accent"]), return_counts=True)
        series = pd.Series(
            counts, index=[WESTBROOK_DATASET_ACCENT_MAP[i] for i in ids], name=split
        )
        li.append(series)
    return pd.concat(li, axis=1).sort_values(by="train", ascending=False)


accent_counts = get_accent_counts()
accent_counts

In [ ]:
model_type = "whisper"
base_model_name_or_path = "openai/whisper-tiny.en"
dataset_path = "westbrook/English_Accent_DataSet"
processor = processor_init(ModelArguments(base_model_name_or_path="openai/whisper-tiny.en"))

In [ ]:
train_dataset = WhisperDataset(
    data_path="westbrook/English_Accent_DataSet",
    split="train",
    processor=processor,
    multilingual_model=False,
)
eval_dataset = WhisperDataset(
    data_path="westbrook/English_Accent_DataSet",
    split="validation",
    processor=processor,
    multilingual_model=False,
)

In [ ]:
from torch.utils.data import DataLoader

# Create data collator
data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

# Create DataLoader
dataloader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True,
    collate_fn=data_collator,
    num_workers=0,  # Set to >0 for multiprocessing
)

# Test the dataloader
batch = next(iter(dataloader))
print("Batch keys:", batch.keys())
print("Input features shape:", batch["input_features"].shape)
print("Labels shape:", batch["labels"].shape)
print("Attention mask shape:", batch["attention_mask"].shape)

In [ ]:
dummy_input = [
    {
        "input_features": np.random.randn(80, 3000),
        "attention_mask": np.ones(3000),
        "labels": [100] * i,
    }
    for i in [100, 1020, 1030]
]

# data_collator(dummy_input)["labels"].shape

In [ ]:
processor.tokenizer.pad(
    [{"input_ids": i["labels"]} for i in dummy_input], return_tensors="pt"
)["input_ids"].shape

In [ ]:
processor.tokenizer.bos_token_id, processor.tokenizer.eos_token_id

In [ ]:
processor.decode([50256, 50257])

In [ ]:
batch["labels"]